# Conditional VAE on CelebA: Generating and Manipulating Faces

In-Class Exercise

---

## The Big Picture

In the MNIST CVAE exercise, we conditioned on **digit labels** (10 mutually exclusive classes). Now we scale up to **CelebA** — real face images conditioned on **40 binary attributes** (Smiling, Eyeglasses, Male, Blond_Hair, etc.).

| | MNIST CVAE | CelebA CVAE |
|---|---|---|
| Image size | 28x28x1 (grayscale) | 64x64x3 (RGB) |
| Condition | 10-class one-hot | 40 binary attributes |
| Latent dim | 32 | 128 |
| Training time | Minutes | ~12 min (30k subset on T4) |

The core concepts (ELBO, reparametrization, beta trade-off) are **identical** — only the scale differs.

### What We'll Do Today
1. Explore CelebA dataset and attribute distributions
2. Build a CelebA Conditional VAE (bigger architecture)
3. Train it on face images with attribute conditions
4. Reconstruct faces and measure quality
5. **Generate faces with specific attributes** (Smiling, Eyeglasses, etc.)
6. Explore the latent space
7. Interpolate between faces and **manipulate attributes**
8. Experiment with the beta hyperparameter

In [ ]:
# @title Setup {display-mode: "form"}

# @markdown Clone the course repo and import helpers. Just run and move on!
# -- Step 1: Clone the course repo ----------------------------------------
%cd /content
![ ! -d 'coding-exercises' ] && git clone https://github.com/eth-bmai-fs26/coding-exercises.git
%cd coding-exercises
!git checkout conditional_vae
!git pull origin conditional_vae
%cd "CVAE CX"

# -- Step 2: Import helpers ------------------------------------------------
from utils import *           # training loops, plotting helpers, DEVICE
from models import *          # CelebAAttributePredictor
from data import *            # load_celeba(), CELEBA_ATTR_NAMES

setup()                       # confirms device (CPU / GPU)

---
## Part 1: Exploring the CelebA Dataset

### From Digits to Faces

CelebA contains over 200,000 celebrity face images, each annotated with **40 binary attributes**. Unlike MNIST's mutually exclusive classes, these attributes are **multi-label** — a face can be both "Smiling" AND "Wearing_Lipstick" AND "Young" simultaneously.

We load a subset (30k images) to fit in Colab's memory. The dataset is downloaded from HuggingFace.

In [ ]:
X_train, attrs_train, X_test, attrs_test = load_celeba(max_samples=30000)

print(f"\nAttribute names ({len(CELEBA_ATTR_NAMES)}):")
for i, name in enumerate(CELEBA_ATTR_NAMES):
    print(f"  {i:2d}. {name}", end="\t" if (i+1) % 4 != 0 else "\n")

In [ ]:
# @title Fallback: install datasets if needed {display-mode: "form"}
# @markdown Uncomment and run ONLY if load_celeba() fails with an import error.

# !pip install -q datasets

In [ ]:
# @title Sample faces from CelebA {display-mode: "form"}

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Sample faces from CelebA", fontsize=14, fontweight='bold')
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(np.clip(X_train[i].transpose(1, 2, 0), 0, 1))
    # Show top 3 active attributes
    active = [CELEBA_ATTR_NAMES[j] for j in range(40) if attrs_train[i, j] == 1.0]
    ax.set_title(', '.join(active[:3]), fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# @title Attribute frequency distribution {display-mode: "form"}

freq = attrs_train.mean(axis=0)
sorted_idx = np.argsort(freq)[::-1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(40), freq[sorted_idx], color='steelblue', edgecolor='black')
ax.set_xticks(range(40))
ax.set_xticklabels([CELEBA_ATTR_NAMES[i] for i in sorted_idx],
                   rotation=90, fontsize=8)
ax.set_ylabel('Fraction of images', fontsize=12)
ax.set_title('CelebA Attribute Frequencies (training set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Most common:  {CELEBA_ATTR_NAMES[sorted_idx[0]]} ({freq[sorted_idx[0]]:.1%})")
print(f"Least common: {CELEBA_ATTR_NAMES[sorted_idx[-1]]} ({freq[sorted_idx[-1]]:.1%})")

---
## Part 2: Understanding the CelebA CVAE Architecture

### Scaling Up from MNIST

The architecture is the same idea as MNIST, but bigger:

| Component | MNIST | CelebA |
|---|---|---|
| Encoder input | (1+10, 28, 28) = (11, 28, 28) | (3+40, 64, 64) = (43, 64, 64) |
| Conv blocks | 3 (stride 2, 2, 1) | 4 (all stride 2) |
| Spatial reduction | 28 -> 14 -> 7 | 64 -> 32 -> 16 -> 8 -> 4 |
| Latent dim | 32 | 128 |
| Decoder input | z(32) + label(10) = 42 | z(128) + attrs(40) = 168 |

**Conditioning mechanism is identical:**
- **Encoder:** broadcast 40 attribute channels spatially, concatenate with image -> (43, 64, 64)
- **Decoder:** concatenate z with attribute vector -> (168,)

### The Loss Function (ELBO)

$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x,c)}[\log p(x|z,c)]}_{\text{Reconstruction (BCE)}} - \underbrace{\beta \cdot D_{KL}\big(q(z|x,c) \| p(z)\big)}_{\text{KL Divergence}}$$

The loss function is **dataset-agnostic** — the same BCE + KL formula works for MNIST (1x28x28) and CelebA (3x64x64).

---
### TODO 1 — Complete the CelebAConvCVAE class

The `__init__` defines all layers. Your task is to fill in the methods:

**`conditional_input(self, x, label)`** — Prepare the encoder input:
1. Reshape `label` from `(B, 40)` to `(B, 40, 1, 1)`
2. Expand to `(B, 40, 64, 64)` using `.expand()`
3. Concatenate with `x` along the channel dimension -> `(B, 43, 64, 64)`

**`reparametrize(self, mu, log_var)`** — The reparametrization trick:
1. Compute `std = exp(0.5 * log_var)`
2. Sample `eps` from N(0, I) with the same shape as `std`
3. Return `mu + std * eps`

**`encode(self, x, label)`** — Encoder forward pass:
1. Build conditional input
2. Pass through `self.encoder_conv`, flatten, then through `self.fc_mu` and `self.fc_log_var`
3. Return `mu, log_var`

**`decode(self, z, label)`** — Decoder forward pass:
1. Concatenate `z` and `label` along dim=1 -> `(B, latent_dim + 40)`
2. Pass through `self.decoder_fc`, reshape to `(B, 512, 4, 4)`, then through `self.decoder_conv`

**`forward(self, x, label)`** — Full CVAE pass:
1. Encode -> reparametrize -> decode
2. Return `recon_x, mu, log_var`

In [ ]:
class CelebAConvCVAE(nn.Module):
    """
    Convolutional Conditional VAE for CelebA.
    Encoder: (43, 64, 64) -> mu, log_var  (latent_dim each)
    Decoder: (latent_dim + 40) -> (3, 64, 64)
    """
    def __init__(self, latent_dim=128, label_dim=40):
        super().__init__()
        self.latent_dim = latent_dim
        self.label_dim = label_dim

        # -- ENCODER --------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3 + label_dim, 64, 4, stride=2, padding=1),    # -> (64, 32, 32)
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),              # -> (128, 16, 16)
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),             # -> (256, 8, 8)
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, stride=2, padding=1),             # -> (512, 4, 4)
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
        )
        self.fc_mu      = nn.Linear(512 * 4 * 4, latent_dim)
        self.fc_log_var = nn.Linear(512 * 4 * 4, latent_dim)

        # -- DECODER --------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 512 * 4 * 4),
            nn.LeakyReLU(0.2),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),    # -> (256, 8, 8)
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),    # -> (128, 16, 16)
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),     # -> (64, 32, 32)
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 3, 4, stride=2, padding=1),       # -> (3, 64, 64)
            nn.Sigmoid(),
        )

    # --- TODO 1a: Conditional input ---
    def conditional_input(self, x, label):
        """Concatenate attribute vector (broadcast spatially) with the image."""
        # SOLUTION
        label_map = label.unsqueeze(-1).unsqueeze(-1)                        # (B, 40, 1, 1)
        label_map = label_map.expand(-1, -1, x.size(2), x.size(3))          # (B, 40, 64, 64)
        return torch.cat([x, label_map], dim=1)                              # (B, 43, 64, 64)

    # --- TODO 1b: Reparametrization trick ---
    def reparametrize(self, mu, log_var):
        """Sample z = mu + std * eps, where eps ~ N(0, I)."""
        # SOLUTION
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    # --- TODO 1c: Encode ---
    def encode(self, x, label):
        """Encode image + attributes -> (mu, log_var)."""
        # SOLUTION
        cond = self.conditional_input(x, label)
        h = self.encoder_conv(cond)
        h = h.flatten(1)
        return self.fc_mu(h), self.fc_log_var(h)

    # --- TODO 1d: Decode ---
    def decode(self, z, label):
        """Decode latent z + attributes -> reconstructed image."""
        # SOLUTION
        z_cond = torch.cat([z, label], dim=1)                                # (B, 168)
        h = self.decoder_fc(z_cond)
        h = h.view(-1, 512, 4, 4)
        return self.decoder_conv(h)

    # --- TODO 1e: Forward pass ---
    def forward(self, x, label):
        """Full CVAE pass: encode -> reparametrize -> decode."""
        # SOLUTION
        mu, log_var = self.encode(x, label)
        z = self.reparametrize(mu, log_var)
        recon_x = self.decode(z, label)
        return recon_x, mu, log_var

In [ ]:
LATENT_DIM = 128
cvae = CelebAConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in cvae.parameters())
print(f"CelebAConvCVAE created: {total_params:,} parameters on {DEVICE}")

---
### TODO 2 — Complete the CVAE loss function

The loss is **identical** to the MNIST version — it operates per-pixel-per-channel and is agnostic to image dimensions.

**Reconstruction loss (BCE):** `F.binary_cross_entropy(recon_x, x, reduction='sum') / batch_size`

**KL divergence:** $D_{KL} = -\frac{1}{2} \sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$

**Total loss:** `recon_loss + beta * kl_loss`

In [ ]:
def cvae_loss(recon_x, x, mu, log_var, beta=1.0):
    """
    Compute the CVAE loss = Reconstruction (BCE) + beta * KL divergence.

    Returns
    -------
    total_loss, recon_loss, kl_loss
    """
    batch_size = x.size(0)

    # TODO 2a: Reconstruction loss (BCE, summed over pixels, averaged over batch)
    # SOLUTION
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum') / batch_size

    # TODO 2b: KL divergence (averaged over batch)
    # SOLUTION
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / batch_size

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

---
### Training the CelebA CVAE

Key hyperparameters:
- `latent_dim = 128` — faces need more capacity than digits
- `beta = 1.0` — standard VAE weighting
- `batch_size = 64` — larger images need smaller batches on GPU
- `epochs = 25` — ~12 minutes on Colab T4

> **Note:** CelebA training takes longer than MNIST. You'll see the loss decrease steadily. Generated faces will be somewhat blurry — this is a well-known VAE limitation that motivates GANs and diffusion models.

In [ ]:
print("Training CelebA Conditional VAE...")
cvae, loss_history = train_cvae_celeba(cvae, cvae_loss, X_train, attrs_train,
                                       epochs=25, batch_size=64, lr=1e-3, beta=1.0)

In [ ]:
plot_losses(loss_history)

You should see:
- **Total loss** decreasing overall
- **Reconstruction loss** decreasing steadily
- **KL loss** increasing initially then stabilising (latent space regularisation)

---
## Part 3: Reconstruction Quality

### How Well Does It Reconstruct Faces?

VAE reconstructions of faces will be **blurrier** than MNIST reconstructions. This is expected — faces have much more high-frequency detail, and the KL regularisation smooths the latent space at the cost of fine details.

In [ ]:
recon = reconstruct_cvae_celeba(cvae, X_test[:100], attrs_test[:100])
plot_original_vs_reconstructed_celeba(X_test[:100], recon,
                                      title="CelebA CVAE: Original vs. Reconstructed")

In [ ]:
# @title Per-attribute reconstruction error {display-mode: "form"}
attr_mse = plot_per_attribute_mse(recon, X_test[:100], attrs_test[:100], CELEBA_ATTR_NAMES)

### Debrief

Faces are recognisable but soft. Rare attributes (Bald, Wearing_Hat) are typically harder to reconstruct because the model sees fewer examples during training.

---
## Part 4: Conditional Generation — Generating Faces on Demand

This is the CelebA CVAE's superpower. We can generate faces with **any combination of attributes** by:
1. Sampling z from N(0, I)
2. Choosing desired attributes (e.g., Smiling + Eyeglasses)
3. Feeding both to the decoder

With 40 binary attributes, there are 2^40 possible combinations — the model must generalise to unseen combos!

---
### TODO 3 — Generate faces conditionally

Use `generate_conditional_celeba(model, attrs_dict, attr_names, n, latent_dim)` to generate faces with specific attributes. Try different combinations and observe the variety.

In [ ]:
# SOLUTION
generate_conditional_celeba(cvae, {"Smiling": 1, "Young": 1}, CELEBA_ATTR_NAMES,
                            n=8, latent_dim=LATENT_DIM)

generate_conditional_celeba(cvae, {"Male": 1, "Eyeglasses": 1}, CELEBA_ATTR_NAMES,
                            n=8, latent_dim=LATENT_DIM)

generate_conditional_celeba(cvae, {"Blond_Hair": 1, "Smiling": 1, "Young": 1}, CELEBA_ATTR_NAMES,
                            n=8, latent_dim=LATENT_DIM)

generate_conditional_celeba(cvae, {"Male": 0, "Heavy_Makeup": 1, "Wearing_Lipstick": 1}, CELEBA_ATTR_NAMES,
                            n=8, latent_dim=LATENT_DIM)

In [ ]:
# @title Attribute combination grid {display-mode: "form"}
combos = [
    {"Smiling": 1},
    {"Eyeglasses": 1},
    {"Male": 1},
    {"Blond_Hair": 1, "Young": 1},
    {"Male": 1, "Bald": 1},
    {"Smiling": 1, "Eyeglasses": 1, "Male": 1},
]
generate_attribute_grid(cvae, CELEBA_ATTR_NAMES, combos,
                        latent_dim=LATENT_DIM, n_per_row=8)

### Debrief

Each row shows faces with the **same attributes** but **different styles** (face shape, skin tone, expression details). The attributes control *what*, and z controls *how*. Some rare attribute combos may look less realistic.

---
## Part 5: Exploring the Latent Space

### What Does the Latent Space Encode?

Since attributes are provided as conditions, the latent space should capture **residual style** — lighting, face shape, pose — not attribute identity. We can verify this by colouring the PCA projection by different attributes.

In [ ]:
# @title Latent space coloured by attribute {display-mode: "form"}
visualise_latent_space_celeba(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                              color_by="Smiling", n_viz=3000)

In [ ]:
# Try another attribute
visualise_latent_space_celeba(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                              color_by="Male", n_viz=3000)

### How to Read This

If the CVAE is well-trained, you should see **significant overlap** between the two groups — because the attribute information is in the condition, not in z. Some separation is normal (attributes correlate with style), but less than an unconditional model.

---
## Part 6: Interpolation and Attribute Manipulation

### Latent Interpolation

We can smoothly morph between two faces by interpolating their latent vectors.

---
### TODO 4 — Interpolate between two faces

Pick two test images. Try with `switch_attrs=False` (fixed attributes) and `switch_attrs=True` (attributes change at midpoint).

In [ ]:
# SOLUTION: Fixed attributes
interpolate_latent_celeba(cvae, X_test, attrs_test, idx1=0, idx2=5, n_steps=10)

In [ ]:
# SOLUTION: Attribute switch at midpoint
interpolate_latent_celeba(cvae, X_test, attrs_test, idx1=0, idx2=5,
                          n_steps=10, switch_attrs=True)

### Attribute Manipulation — The Fun Part!

This is unique to CelebA. We can take a real face, encode it, and decode with **modified attributes** — adding or removing a smile, glasses, etc. The latent z preserves the identity while the attributes change the appearance.

In [ ]:
# Try manipulating different attributes on different faces
for attr in ["Smiling", "Eyeglasses", "Male", "Bangs", "Blond_Hair"]:
    manipulate_attribute(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                         idx=0, attr_to_change=attr)

In [ ]:
# Try on a different face
for attr in ["Smiling", "Eyeglasses", "Young"]:
    manipulate_attribute(cvae, X_test, attrs_test, CELEBA_ATTR_NAMES,
                         idx=10, attr_to_change=attr)

### Debrief

- **Fixed attributes:** the face smoothly morphs from source to target style while keeping the same attributes
- **Attribute switch:** the face structure morphs smoothly but attributes change abruptly at midpoint
- **Attribute manipulation:** the identity is roughly preserved while the specified attribute changes

This demonstrates the **disentanglement** between content (attributes) and style (z).

---
## Part 7: The Effect of Beta

| Beta | Reconstruction | Latent Space | Generation |
|---|---|---|---|
| Low (0.1) | Sharp, detailed | Irregular | May produce artifacts |
| Standard (1.0) | Good balance | Regular | Decent quality |
| High (5.0) | Blurry | Very regular | Diverse but fuzzy |

---
### TODO 5 — Experiment with different beta values

Train the CelebA CVAE with different beta values. For each, generate an attribute grid and compare.

In [ ]:
# SOLUTION
test_combos = [
    {"Smiling": 1},
    {"Eyeglasses": 1},
    {"Male": 1, "Young": 1},
]

for beta_val in [0.1, 1.0, 5.0]:
    print(f"\n{'='*50}")
    print(f"Training with beta = {beta_val}")
    print(f"{'='*50}")
    model_beta = CelebAConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
    model_beta, _ = train_cvae_celeba(model_beta, cvae_loss, X_train, attrs_train,
                                      epochs=15, batch_size=64, lr=1e-3, beta=beta_val)
    generate_attribute_grid(model_beta, CELEBA_ATTR_NAMES, test_combos,
                            latent_dim=LATENT_DIM, n_per_row=6)

### Debrief

- **beta = 0.1**: sharper faces, but may show attribute leakage (e.g., "Smiling" might not always work)
- **beta = 1.0**: good balance of quality and attribute control
- **beta = 5.0**: blurry but very consistent attribute conditioning

The blurriness at high beta is more noticeable on faces than digits — this is the **rate-distortion trade-off** in action.

---
## Part 8: Evaluating Generation Quality

We train an attribute predictor on real CelebA images, then test it on generated images. If the predictor correctly identifies the attributes we conditioned on, the CVAE is generating attribute-faithful faces.

In [ ]:
# @title Train attribute predictor and evaluate {display-mode: "form"}
print("Training attribute predictor on real CelebA images...")
predictor = train_attribute_predictor(CelebAAttributePredictor, X_train, attrs_train, epochs=10)

print("\nEvaluating CVAE-generated images...")
per_attr_acc = evaluate_generated_celeba(cvae, predictor, CELEBA_ATTR_NAMES,
                                         latent_dim=LATENT_DIM, n_samples=1000)

### Debrief

Common attributes (Smiling, Male, Young) should have higher accuracy than rare ones (Bald, Wearing_Hat). The mean accuracy gives an overall picture of how well the CVAE respects conditioning.

---
## Summary

| Step | What You Did |
|---|---|
| Data exploration | Loaded CelebA with 40 binary attributes |
| Architecture | Built CelebAConvCVAE (43->64->128->256->512, latent=128) |
| TODO 1 | Implemented conditional_input, reparametrize, encode, decode, forward |
| TODO 2 | Implemented CVAE loss (BCE + beta x KL) — same as MNIST |
| Training | Trained on 30k faces with cosine LR schedule |
| Reconstruction | Measured per-attribute reconstruction quality |
| TODO 3 | Generated faces with specific attributes |
| Latent space | Visualised attribute (dis)entanglement |
| TODO 4 | Interpolated between faces, explored attribute switching |
| Attribute manipulation | Flipped attributes while preserving identity |
| TODO 5 | Investigated beta trade-off on faces |
| Evaluation | Tested attribute fidelity with a predictor |

### Key Takeaways
- The **same CVAE architecture** scales from MNIST to CelebA — only dimensions change
- CelebA uses **multi-label binary conditions** (40 attributes) vs MNIST's **one-hot classes** (10 digits)
- **Attribute manipulation** (encode, flip attribute, decode) is the CelebA CVAE's killer feature
- VAE face reconstructions are **blurry** — motivating GANs and diffusion models
- The **beta trade-off** is even more visible on complex data like faces